# 07 · Kvasir-SEG: EIoU vs AEIoU(λ=0.5) — Formula Verification Run

**Purpose:** Verify the corrected AEIoU normalisation (enclosing-box dims cw²/ch² instead of target dims tw²/th²)  
**Losses:** EIoU + AEIoU(λ=0.5) only  
**Split:** Clean only  
**Seeds:** 42, 123, 456  
**Epochs:** 30  
**Dataset:** Kvasir-SEG at `/content/amorphous-yolo/datasets/kvasir-seg`

**TP definition:** A predicted box is a True Positive when its IoU with a matched ground-truth box ≥ **0.50** (COCO/Ultralytics default confusion-matrix threshold).  
NMS IoU threshold (suppresses duplicate boxes): **0.60**  
Confidence threshold for evaluation: **0.001** (standard COCO practice — recall all detections)

Results saved to: `experiments_kvasir_v2/` (separate from prior runs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('GPU:', os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip())

In [ ]:
!pip install --upgrade pip -q
!pip install -U "ultralytics==8.4.9" "wandb==0.24.1" "pycocotools" -q
print('Dependencies installed.')

In [ ]:
import os, sys
REPO_PATH = '/content/amorphous-yolo'
if not os.path.exists(f'{REPO_PATH}/.git'):
    os.system(f'git clone https://github.com/nipun-taneja/amorphous-yolo.git {REPO_PATH}')
else:
    os.system(f'git -C {REPO_PATH} pull --rebase')
    print('Repo pulled.')
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f'CWD: {os.getcwd()}')

In [ ]:
import time
from pathlib import Path
from datetime import datetime

PROJECT_DIR   = Path('/content/amorphous-yolo')
DATASET_ROOT  = PROJECT_DIR / 'datasets' / 'kvasir-seg'
EXPERIMENTS   = PROJECT_DIR / 'experiments_kvasir_v2'      # separate from prior runs
ANALYSIS_DIR  = EXPERIMENTS / 'analysis'
MANIFEST_PATH = EXPERIMENTS / 'manifest.json'
DRIVE_EXP     = Path('/content/drive/MyDrive/amorphous_yolo/experiments_kvasir_v2')
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS   = 30
IMGSZ    = 640
DEVICE   = 0
MODEL_PT = 'yolo26n.pt'
SEEDS    = [42, 123, 456]

LOSS_KEYS = ['eiou', 'aeiou_r0p5']
SPLIT_CONFIGS = {'clean': PROJECT_DIR / 'data' / 'kvasir_seg.yaml'}

PALETTE = {'eiou': '#E63946', 'aeiou_r0p5': '#00B4D8'}
LABELS  = {'eiou': 'EIoU', 'aeiou_r0p5': 'AEIoU (lambda=0.5, corrected)'}

print(f'Runs planned: {len(LOSS_KEYS)} losses x {len(SEEDS)} seeds = {len(LOSS_KEYS)*len(SEEDS)}')
print(f'Epochs: {EPOCHS}  |  Split: clean only')

In [ ]:
import re
from pathlib import Path

(PROJECT_DIR / 'data').mkdir(exist_ok=True)
YAMLS_DIR = EXPERIMENTS / 'yamls'
YAMLS_DIR.mkdir(exist_ok=True)

patched = {}
for split_name, orig in SPLIT_CONFIGS.items():
    if not Path(orig).exists():
        # Write a minimal YAML if it doesn't exist yet
        content = f'path: {DATASET_ROOT}\ntrain: train/images\nval: valid/images\nnc: 1\nnames: [polyp]\n'
    else:
        content = Path(orig).read_text()
        content = re.sub(r'^path:.*$', f'path: {DATASET_ROOT}', content, flags=re.MULTILINE)
    out = YAMLS_DIR / Path(orig).name
    out.write_text(content)
    patched[split_name] = out
    print(f'YAML {split_name}: {out}  ->  path={DATASET_ROOT}')

SPLIT_CONFIGS = patched

In [ ]:
import os, wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'), quiet=True)
    print('WandB ready.')
except Exception as e:
    os.environ['WANDB_MODE'] = 'disabled'
    print(f'WandB disabled ({e})')

In [ ]:
# Dataset is expected at /content/amorphous-yolo/datasets/kvasir-seg
# If train/valid dirs are missing, download + convert masks (idempotent).
import cv2, numpy as np, random, shutil
from pathlib import Path

TRAIN_DIR = DATASET_ROOT / 'train'
VALID_DIR = DATASET_ROOT / 'valid'

n_train = len(list((TRAIN_DIR/'images').glob('*.jpg'))) if (TRAIN_DIR/'images').exists() else 0
n_valid = len(list((VALID_DIR/'images').glob('*.jpg'))) if (VALID_DIR/'images').exists() else 0
print(f'Found: {n_train} train  {n_valid} valid')

if n_train < 800 or n_valid < 100:
    print('Dataset incomplete — downloading and converting...')
    import zipfile, urllib.request
    RAW_DIR = DATASET_ROOT / 'raw'
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    ZIP = DATASET_ROOT / 'kvasir-seg.zip'
    if not ZIP.exists():
        print('Downloading Kvasir-SEG (~46 MB)...')
        urllib.request.urlretrieve('https://datasets.simula.no/downloads/kvasir-seg.zip', ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(RAW_DIR)
    RAW_IMG  = next(RAW_DIR.rglob('images'), None) or RAW_DIR
    RAW_MASK = next(RAW_DIR.rglob('masks'),  None) or RAW_DIR

    def _mask_to_bbox(p):
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if m is None: return None
        _, b = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
        cs, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cs: return None
        x,y,w,h = cv2.boundingRect(np.vstack(cs))
        H,W = m.shape
        return float(np.clip((x+w/2)/W,0,1)), float(np.clip((y+h/2)/H,0,1)), float(np.clip(w/W,0,1)), float(np.clip(h/H,0,1))

    imgs = sorted(RAW_IMG.glob('*.jpg'))
    random.seed(42); random.shuffle(imgs)
    split_idx = int(0.8*len(imgs))
    for sname, img_list in [('train', imgs[:split_idx]), ('valid', imgs[split_idx:])]:
        (DATASET_ROOT/sname/'images').mkdir(parents=True, exist_ok=True)
        (DATASET_ROOT/sname/'labels').mkdir(parents=True, exist_ok=True)
        for ip in img_list:
            mp = RAW_MASK / ip.name
            if not mp.exists(): mp = RAW_MASK / (ip.stem + '.png')
            b = _mask_to_bbox(mp)
            if b is None: continue
            shutil.copy2(ip, DATASET_ROOT/sname/'images'/ip.name)
            (DATASET_ROOT/sname/'labels'/f'{ip.stem}.txt').write_text(f'0 {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n')
    n_train = len(list((TRAIN_DIR/'images').glob('*.jpg')))
    n_valid = len(list((VALID_DIR/'images').glob('*.jpg')))

assert n_train >= 800 and n_valid >= 100, f'Dataset check failed: train={n_train} valid={n_valid}'
print(f'Dataset OK: {n_train} train  {n_valid} valid')

In [ ]:
# FORMULA FIX — applied at runtime without touching src/losses.py
# Bug: original AEIoU normalised size errors by tw^2/th^2 (target dims)
# Fix: normalise by cw^2/ch^2 (enclosing-box dims), matching EIoU's convention
# This makes AEIoU(lambda=1.0) == EIoU exactly, AEIoU(lambda=0) == DIoU exactly.
import torch
from src.losses import AEIoULoss, EIoULoss

def _aeiou_forward_corrected(self, pred_boxes, target_boxes):
    eps = self.eps
    px1,py1,px2,py2 = pred_boxes.unbind(-1)
    tx1,ty1,tx2,ty2 = target_boxes.unbind(-1)
    pw=(px2-px1).clamp(min=eps); ph=(py2-py1).clamp(min=eps)
    tw=(tx2-tx1).clamp(min=eps); th=(ty2-ty1).clamp(min=eps)
    pcx=(px1+px2)*0.5; pcy=(py1+py2)*0.5
    tcx=(tx1+tx2)*0.5; tcy=(ty1+ty2)*0.5
    # Intersection + IoU
    ix1=torch.max(px1,tx1); iy1=torch.max(py1,ty1)
    ix2=torch.min(px2,tx2); iy2=torch.min(py2,ty2)
    inter=(ix2-ix1).clamp(min=0)*(iy2-iy1).clamp(min=0)
    union=pw*ph+tw*th-inter+eps
    iou=inter/union
    # Enclosing box
    cx1=torch.min(px1,tx1); cy1=torch.min(py1,ty1)
    cx2=torch.max(px2,tx2); cy2=torch.max(py2,ty2)
    cw=(cx2-cx1).clamp(min=eps); ch=(cy2-cy1).clamp(min=eps)
    c2=cw*cw+ch*ch
    # DIoU center term
    diou_term=((pcx-tcx)**2+(pcy-tcy)**2)/(c2+eps)
    # Size term: CORRECTED — enclosing dims cw^2/ch^2 (NOT target dims tw^2/th^2)
    w_err2=(pw-tw)**2/(cw*cw+eps)
    h_err2=(ph-th)**2/(ch*ch+eps)
    loss=1.0-iou+diou_term+self.rigidity*(w_err2+h_err2)
    if self.reduction=='mean': return loss.mean()
    if self.reduction=='sum':  return loss.sum()
    return loss

AEIoULoss.forward = _aeiou_forward_corrected
print('AEIoU.forward patched: size term normalised by cw^2/ch^2 (enclosing box)')

# === UNIT TEST: AEIoU(lambda=1.0) must equal EIoU ===
torch.manual_seed(0)
p = torch.rand(50,4); p[:,2:]+=p[:,:2]+0.05  # valid xyxy boxes
g = torch.rand(50,4); g[:,2:]+=g[:,:2]+0.05
e_loss = EIoULoss(reduction='none')(p,g)
a_loss = AEIoULoss(rigidity=1.0, reduction='none')(p,g)
max_diff = (e_loss-a_loss).abs().max().item()
print(f'Unit test AEIoU(1.0) vs EIoU  max_diff={max_diff:.2e}  (must be < 1e-5)')
assert max_diff < 1e-5, f'FAIL: formula still wrong, max_diff={max_diff}'
print('PASS: AEIoU(lambda=1.0) == EIoU confirmed with corrected formula.')

In [ ]:
import torch.nn.functional as F
import ultralytics.utils.loss as loss_mod

_ORIGINAL_BBOX_FORWARD = loss_mod.BboxLoss.forward

def _make_bbox_forward(loss_fn):
    def bbox_loss_forward(self, pred_dist, pred_bboxes, anchor_points,
                          target_bboxes, target_scores, target_scores_sum,
                          fg_mask, imgsz, stride):
        weight   = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        per_box  = loss_fn(pred_bboxes[fg_mask], target_bboxes[fg_mask])
        loss_iou = (per_box.unsqueeze(-1)*weight).sum()/target_scores_sum
        if self.dfl_loss:
            target_ltrb = loss_mod.bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max-1)
            loss_dfl = self.dfl_loss(pred_dist[fg_mask].view(-1,self.dfl_loss.reg_max), target_ltrb[fg_mask])*weight
            loss_dfl = loss_dfl.sum()/target_scores_sum
        else:
            tl = loss_mod.bbox2dist(anchor_points,target_bboxes)*stride
            tl[...,0::2]/=imgsz[1]; tl[...,1::2]/=imgsz[0]
            pd = pred_dist*stride
            pd[...,0::2]/=imgsz[1]; pd[...,1::2]/=imgsz[0]
            loss_dfl = (F.l1_loss(pd[fg_mask],tl[fg_mask],reduction='none').mean(-1,keepdim=True)*weight).sum()/target_scores_sum
        return loss_iou, loss_dfl
    return bbox_loss_forward

def patch_loss(fn):   loss_mod.BboxLoss.forward = _make_bbox_forward(fn)
def restore_loss():   loss_mod.BboxLoss.forward = _ORIGINAL_BBOX_FORWARD

# Verify round-trip
from src.losses import EIoULoss as _E
patch_loss(_E(reduction='none')); restore_loss()
assert loss_mod.BboxLoss.forward is _ORIGINAL_BBOX_FORWARD
print('Monkey-patch round-trip OK.')

In [ ]:
from src.losses import EIoULoss, AEIoULoss

LOSS_REGISTRY = {
    'eiou':      EIoULoss(reduction='none'),
    'aeiou_r0p5': AEIoULoss(rigidity=0.5, reduction='none'),
}
for k, fn in LOSS_REGISTRY.items():
    r = getattr(fn,'rigidity',None)
    print(f'  {k}: {type(fn).__name__}' + (f'  rigidity={r}' if r else ''))

In [ ]:
import json, shutil
from ultralytics import YOLO
from pathlib import Path as _P

def _manifest():
    return json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}

def _save_manifest(m):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2))

def _sync(run_name):
    try:
        DRIVE_EXP.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(EXPERIMENTS/run_name), str(DRIVE_EXP/run_name), dirs_exist_ok=True)
    except Exception as e:
        print(f'  [DRIVE] sync failed: {e}')

def run_training(loss_name, loss_fn, split_name, yaml_path, seed=42):
    run_name = f'kvasir_v2_{loss_name}_{split_name}_s{seed}_e{EPOCHS}'
    run_dir  = EXPERIMENTS / run_name
    csv      = run_dir / 'results.csv'
    last_pt  = run_dir / 'weights' / 'last.pt'

    # Session-drop recovery
    if csv.exists():
        import pandas as _pd
        n = len(_pd.read_csv(csv))
        if n >= EPOCHS:
            print(f'[SKIP] {run_name}  ({n}/{EPOCHS} epochs)'); return run_dir
        resuming = last_pt.exists()
        if resuming: print(f'[PARTIAL] {n}/{EPOCHS} epochs -- resuming')
        else:        print(f'[RESTART] {n}/{EPOCHS} partial, no last.pt')
    else:
        # Restore from Drive if available
        drive_run = DRIVE_EXP / run_name
        if drive_run.exists() and not run_dir.exists():
            shutil.copytree(str(drive_run), str(run_dir), dirs_exist_ok=True)
            print(f'[RESTORED] {run_name} from Drive')
        resuming = last_pt.exists()

    print(f"\n{'='*65}")
    print(f"[{'RESUME' if resuming else 'START '}] {run_name}")
    print(f"{'='*65}")

    m = _manifest()
    m[run_name] = {'loss':loss_name,'split':split_name,'seed':seed,'epochs':EPOCHS,
                   'rigidity':float(getattr(loss_fn,'rigidity',-1) or -1),
                   'status':'running','timestamp':datetime.now().isoformat()}
    _save_manifest(m)
    t0 = time.time()
    try:
        patch_loss(loss_fn)
        if resuming:
            results = YOLO(str(last_pt)).train(resume=True)
        else:
            results = YOLO(MODEL_PT).train(
                data=str(yaml_path), epochs=EPOCHS, imgsz=IMGSZ,
                project=str(EXPERIMENTS), name=run_name,
                device=DEVICE, seed=seed, exist_ok=True)
        m[run_name]['status'] = 'complete'
        m[run_name]['elapsed_sec'] = round(time.time()-t0,1)
        try:
            import wandb as _w
            if _w.run: _w.finish()
        except: pass
    except Exception as e:
        m[run_name]['status'] = 'failed'; m[run_name]['error'] = str(e)
        raise
    finally:
        restore_loss(); _save_manifest(m); _sync(run_name)
    print(f'[DONE] {run_name}  ({m[run_name].get("elapsed_sec",0):.0f}s)')
    return run_dir

print('run_training() ready.')

In [ ]:
total = len(LOSS_KEYS)*len(SEEDS)
done  = 0
for loss_name in LOSS_KEYS:
    for seed in SEEDS:
        done += 1
        print(f'\n[{done}/{total}] {loss_name}  seed={seed}')
        run_training(
            loss_name = loss_name,
            loss_fn   = LOSS_REGISTRY[loss_name],
            split_name= 'clean',
            yaml_path = SPLIT_CONFIGS['clean'],
            seed      = seed,
        )
restore_loss()
print(f'\nAll {total} runs complete.')

In [ ]:
import pandas as pd, numpy as np

MAP50 = 'metrics/mAP50(B)'
MAP95 = 'metrics/mAP50-95(B)'

dfs = []
for ln in LOSS_KEYS:
    for seed in SEEDS:
        rn  = f'kvasir_v2_{ln}_clean_s{seed}_e{EPOCHS}'
        csv = EXPERIMENTS / rn / 'results.csv'
        if csv.exists():
            df = pd.read_csv(csv); df.columns = df.columns.str.strip()
            df['loss'] = ln; df['seed'] = seed
            dfs.append(df)
        else:
            print(f'  [MISSING] {rn}')

df_all = pd.concat(dfs, ignore_index=True)
df_fin = df_all.groupby(['loss','seed']).last().reset_index()

print('=== Final-epoch results (per seed) ===')
print(df_fin[['loss','seed',MAP50,MAP95]].round(4).to_string(index=False))

print('\n=== Mean +/- Std over 3 seeds ===')
agg = df_fin.groupby('loss')[[MAP50,MAP95]].agg(['mean','std'])
for ln in LOSS_KEYS:
    m50  = agg.loc[ln,(MAP50,'mean')];  s50 = agg.loc[ln,(MAP50,'std')]
    m95  = agg.loc[ln,(MAP95,'mean')];  s95 = agg.loc[ln,(MAP95,'std')]
    delta95 = m95 - agg.loc['eiou',(MAP95,'mean')] if ln != 'eiou' else 0.0
    print(f'  {LABELS[ln]}')
    print(f'    mAP50:    {m50:.4f} +/- {s50:.4f}')
    print(f'    mAP50-95: {m95:.4f} +/- {s95:.4f}' + (f'  (delta vs EIoU: {delta95:+.4f})' if ln!='eiou' else ''))

agg.to_csv(ANALYSIS_DIR/'summary_table.csv')
df_fin.to_csv(ANALYSIS_DIR/'per_seed_final.csv', index=False)
print('\nSaved summary_table.csv and per_seed_final.csv')

In [ ]:
# Cell A: Build COCO-format GT JSON from YOLO val labels (idempotent)
# Required by pycocotools to compute APs/APm/APl.
import cv2, json as _json, hashlib
from pathlib import Path

def _img_id(stem):
    if stem.isdigit(): return int(stem)
    return int(hashlib.md5(stem.encode()).hexdigest(),16) % (2**31)

def build_coco_gt(val_img_dir, val_lbl_dir, out_path):
    if Path(out_path).exists():
        print(f'COCO GT exists: {out_path}'); return
    images, anns, ann_id = [], [], 1
    for ip in sorted(Path(val_img_dir).glob('*.jpg')):
        img = cv2.imread(str(ip))
        if img is None: continue
        H,W = img.shape[:2]
        iid = _img_id(ip.stem)
        images.append({'id':iid,'file_name':ip.name,'width':W,'height':H})
        lbl = Path(val_lbl_dir)/f'{ip.stem}.txt'
        if not lbl.exists(): continue
        for line in lbl.read_text().strip().splitlines():
            if not line.strip(): continue
            _,cx,cy,bw,bh = map(float,line.split())
            x1=(cx-bw/2)*W; y1=(cy-bh/2)*H; wa=bw*W; ha=bh*H
            anns.append({'id':ann_id,'image_id':iid,'category_id':1,
                         'bbox':[x1,y1,wa,ha],'area':wa*ha,'iscrowd':0})
            ann_id += 1
    Path(out_path).write_text(_json.dumps(
        {'images':images,'annotations':anns,'categories':[{'id':1,'name':'polyp'}]}))
    print(f'COCO GT: {out_path}  ({len(images)} imgs, {len(anns)} anns)')

COCO_GT = DATASET_ROOT/'valid'/'coco_gt.json'
build_coco_gt(DATASET_ROOT/'valid'/'images', DATASET_ROOT/'valid'/'labels', COCO_GT)

In [ ]:
# Cell B: compute_and_persist_metrics
# TP threshold: IoU >= 0.50 with matched GT box (COCO default, Ultralytics confusion matrix default)
# NMS IoU: 0.60 (suppresses duplicate detections only, does not affect TP definition)
# Conf for eval: 0.001 (recalls all detections for PR curve computation)
import io, contextlib, json as _json, numpy as np
from pathlib import Path
from ultralytics.cfg import get_cfg, DEFAULT_CFG
from ultralytics.models.yolo.detect import DetectionValidator

METRICS_DIR = EXPERIMENTS/'metrics'
METRICS_DIR.mkdir(exist_ok=True)

def compute_metrics(run_name, weights, yaml_path, force=False):
    out  = METRICS_DIR/run_name
    done = out/'coco_ap_suite.json'
    if done.exists() and not force:
        print(f'  [SKIP] {run_name}'); return
    if not Path(weights).exists():
        print(f'  [MISS] {weights}'); return
    out.mkdir(exist_ok=True)

    args = get_cfg(DEFAULT_CFG, dict(model=str(weights), data=str(yaml_path),
                   conf=0.001, iou=0.6, verbose=False, save_json=True, imgsz=640, split='val'))
    v = DetectionValidator(args=args); v()
    box = v.metrics.box

    # PR curve
    pr = {'precision': np.atleast_2d(box.p)[0].tolist(),
          'recall':    np.atleast_2d(box.r)[0].tolist(),
          'f1':        np.atleast_2d(box.f1)[0].tolist(),
          'map50': float(box.map50), 'map75': float(box.map75),
          'map50_95': float(box.map),
          'iou_thresholds': np.round(np.arange(0.5,1.0,0.05),2).tolist(),
          'tp_iou_threshold': 0.50,   # TP when pred-GT IoU >= this value
          'nms_iou_threshold': 0.60,  # NMS suppresses duplicates above this
          'conf_threshold_eval': 0.001}
    (out/'pr_curve.json').write_text(_json.dumps(pr,indent=2))

    # COCO suite via pycocotools
    suite = {'map50_95':float(box.map),'map50':float(box.map50),'map75':float(box.map75),
             'APs':None,'APm':None,'APl':None,'AR_1':None,'AR_10':None,'AR_100':None}
    cands = list(EXPERIMENTS.glob('**/*predictions*.json'))
    pred_json = max(cands,key=lambda p:p.stat().st_mtime) if cands else None
    if pred_json:
        try:
            raw = _json.loads(pred_json.read_text())
            remap = [{**p,'image_id':(_img_id(str(p['image_id'])) if not isinstance(p['image_id'],int) else p['image_id'])} for p in raw]
            fixed = out/'pred_fixed.json'; fixed.write_text(_json.dumps(remap))
            from pycocotools.coco import COCO; from pycocotools.cocoeval import COCOeval
            gt = COCO(str(COCO_GT)); dt = gt.loadRes(str(fixed))
            ev = COCOeval(gt,dt,'bbox')
            buf=io.StringIO()
            with contextlib.redirect_stdout(buf): ev.evaluate(); ev.accumulate(); ev.summarize()
            s=ev.stats
            suite.update({'map50_95':float(s[0]),'map50':float(s[1]),'map75':float(s[2]),
                          'APs':float(s[3]),'APm':float(s[4]),'APl':float(s[5]),
                          'AR_1':float(s[6]),'AR_10':float(s[7]),'AR_100':float(s[8])})
        except Exception as e:
            print(f'  [WARN] pycocotools: {e}')
    (out/'coco_ap_suite.json').write_text(_json.dumps(suite,indent=2))

    # Confusion matrix (TP/FP/FN) — IoU threshold = 0.50
    try:
        cm=v.confusion_matrix; mat=cm.matrix.tolist()
        cf={'matrix':mat,'tp_iou_threshold':0.50,
            'TP':float(mat[0][0]),'FN':float(mat[0][1]) if len(mat[0])>1 else None,
            'FP':float(mat[1][0]) if len(mat)>1 else None}
    except Exception as e:
        cf={'error':str(e)}
    (out/'confusion_matrix.json').write_text(_json.dumps(cf,indent=2))

    print(f'  [DONE] {run_name}  mAP95={suite["map50_95"]:.4f}  mAP50={suite["map50"]:.4f}  TP={cf.get("TP")}  FP={cf.get("FP")}  FN={cf.get("FN")}')

print('compute_metrics() ready.')
print('NOTE: TP = pred-GT IoU >= 0.50  |  NMS IoU = 0.60  |  Eval conf = 0.001')

In [ ]:
# Cell C: run metrics for all completed runs
for ln in LOSS_KEYS:
    for seed in SEEDS:
        rn = f'kvasir_v2_{ln}_clean_s{seed}_e{EPOCHS}'
        w  = EXPERIMENTS/rn/'weights'/'best.pt'
        compute_metrics(rn, w, SPLIT_CONFIGS['clean'])

# Aggregate confusion table
import json as _json, pandas as pd
rows = []
for ln in LOSS_KEYS:
    for seed in SEEDS:
        rn = f'kvasir_v2_{ln}_clean_s{seed}_e{EPOCHS}'
        cf_path = METRICS_DIR/rn/'confusion_matrix.json'
        ap_path = METRICS_DIR/rn/'coco_ap_suite.json'
        if cf_path.exists() and ap_path.exists():
            cf = _json.loads(cf_path.read_text())
            ap = _json.loads(ap_path.read_text())
            rows.append({'loss':LABELS[ln],'seed':seed,
                         'mAP50':ap['map50'],'mAP50-95':ap['map50_95'],
                         'TP':cf.get('TP'),'FP':cf.get('FP'),'FN':cf.get('FN'),
                         'tp_iou_threshold':cf.get('tp_iou_threshold',0.50)})
df_cm = pd.DataFrame(rows)
print('\n=== Confusion Metrics (TP threshold: IoU >= 0.50) ===')
print(df_cm.round(4).to_string(index=False))
df_cm.to_csv(ANALYSIS_DIR/'confusion_metrics.csv', index=False)

In [ ]:
# Cell D: full PR-curve sweep CSVs (precision/recall at every conf x IoU threshold)
# Output schema: loss, iou_threshold, conf_threshold, precision, recall, f1, seed
# IoU thresholds: 0.50..0.95 step 0.05 (Ultralytics' fixed eval grid)
# Conf thresholds: ~1000 points from ap_per_class' internal sweep (per IoU bin)
from src.pr_curves import export_pr_curves_csv
import pandas as pd

PR_CSV_DIR = EXPERIMENTS / 'pr_curves'
PR_CSV_DIR.mkdir(exist_ok=True)

frames = []
for ln in LOSS_KEYS:
    for seed in SEEDS:
        rn      = f'kvasir_v2_{ln}_clean_s{seed}_e{EPOCHS}'
        run_dir = EXPERIMENTS / rn
        weights = run_dir / 'weights' / 'best.pt'
        if not weights.exists():
            print(f'  [MISS] {rn}'); continue
        out_csv = PR_CSV_DIR / f'{rn}_pr_curve.csv'
        export_pr_curves_csv(
            run_dir=run_dir,
            data_yaml=SPLIT_CONFIGS['clean'],
            loss_name=ln,
            out_csv=out_csv,
            conf=0.001, iou_nms=0.6, imgsz=IMGSZ, device=DEVICE,
        )
        df = pd.read_csv(out_csv)
        df['seed'] = seed
        frames.append(df)
        print(f'  [DONE] {rn}  rows={len(df)}')

if frames:
    master = pd.concat(frames, ignore_index=True)
    master_path = EXPERIMENTS / 'pr_curves_all_losses.csv'
    master.to_csv(master_path, index=False)
    print(f'\nmaster CSV: {master_path}  rows={len(master)}')
    print('Row counts by (loss, iou_threshold, seed):')
    print(master.groupby(['loss','iou_threshold','seed']).size().head(12))


In [ ]:
import matplotlib.pyplot as plt, numpy as np, json as _json

fig, ax = plt.subplots(figsize=(8,6))
fig.suptitle('Kvasir-SEG: PR Curves — EIoU vs AEIoU(lambda=0.5, corrected)\n'
             'Averaged over seeds [42,123,456] | Eval conf=0.001 | TP: IoU>=0.50', fontsize=10)

for ln in LOSS_KEYS:
    all_prec, all_rec = [], []
    for seed in SEEDS:
        rn  = f'kvasir_v2_{ln}_clean_s{seed}_e{EPOCHS}'
        pr  = METRICS_DIR/rn/'pr_curve.json'
        if pr.exists():
            d = _json.loads(pr.read_text())
            all_prec.append(d['precision'])
            all_rec.append(d['recall'])
    if not all_prec: continue
    min_len = min(len(p) for p in all_prec)
    prec_arr = np.array([p[:min_len] for p in all_prec])
    rec_arr  = np.array([r[:min_len] for r in all_rec])
    mean_p   = prec_arr.mean(axis=0)
    mean_r   = rec_arr.mean(axis=0)
    std_p    = prec_arr.std(axis=0)
    # Sort by recall for clean curve
    idx = np.argsort(mean_r)
    mr, mp, sp = mean_r[idx], mean_p[idx], std_p[idx]
    ax.plot(mr, mp, color=PALETTE[ln], lw=2, label=LABELS[ln])
    ax.fill_between(mr, np.clip(mp-sp,0,1), np.clip(mp+sp,0,1),
                    color=PALETTE[ln], alpha=0.15)

ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR/'pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pr_curves.png')

In [ ]:
import shutil
DRIVE_EXP.mkdir(parents=True, exist_ok=True)

# Sync analysis outputs
drive_analysis = DRIVE_EXP/'analysis'
shutil.copytree(str(ANALYSIS_DIR), str(drive_analysis), dirs_exist_ok=True)

# Sync metrics
drive_metrics = DRIVE_EXP/'metrics'
shutil.copytree(str(METRICS_DIR), str(drive_metrics), dirs_exist_ok=True)

# Sync manifest
if MANIFEST_PATH.exists():
    shutil.copy2(str(MANIFEST_PATH), str(DRIVE_EXP/'manifest.json'))

print(f'All outputs synced to Drive: {DRIVE_EXP}')
print('Files:')
for f in sorted(drive_analysis.glob('*')):
    print(f'  analysis/{f.name}  ({f.stat().st_size//1024} KB)')